In [1]:
from datasets import load_dataset
import json
from datasets import Dataset, DatasetDict
from qiskit.qasm3 import loads, dumps
from qiskit import transpile

DATASET_NAME = "valterUo/graph-data-quantum-updated"
OUTPUT_NAME = "valterUo/graph-data-quantum-optimized"

ds = load_dataset(DATASET_NAME)
updated_data = None
ds_new = {}

with open("optimization_results\optimization_results_train.json") as f:
    updated_data_train = json.load(f)

with open("optimization_results\optimization_results_test.json") as f:
    updated_data_test = json.load(f)

# Create new dataset structure
ds_new = {}

for split in ds.keys():
    ds_new[split] = []
    split_data = ds[split]
    
    # Choose the appropriate updated data based on split
    updated_data = updated_data_train if split == 'train' else updated_data_test
    
    for entry in split_data:
        signature = entry['signature']
        
        # Find matching entry in optimization results
        matching_item = None
        for key, item in updated_data.items():
            if item['signature'] == signature:
                matching_item = item
                break
        
        if matching_item:
            # Create new entry with updated fields
            new_entry = entry.copy()  # Keep all original fields
            circuit = loads(matching_item["final_circuit_qasm"])
            basis_gates = ['rx', 'ry', 'rz', 'cx', 'h', 'cz', 'swap', 'sx', 'sxdg', 't', 'tdg', 'id', 'x', 'y', 'z']
            transpiled_circuit = transpile(
                circuit,
                basis_gates=basis_gates,
                optimization_level=1
            )
            new_entry["circuit_with_params"] = str(dumps(transpiled_circuit))
            new_entry["improvement_ratio"] = matching_item["improvement_ratio"]
            new_entry["optimized_params"] = matching_item["optimized_params"]
            new_entry["optimization_info"] = matching_item["optimization_info"]
            new_entry["total_iterations"] = matching_item["total_iterations"]
            ds_new[split].append(new_entry)
        else:
            print(f"Warning: No matching entry found for signature {signature} in {split} set.")
            # Keep original entry if no match found
            ds_new[split].append(entry)

# Convert to Dataset format
for split in ds_new.keys():
    ds_new[split] = Dataset.from_list(ds_new[split])

ds_new = DatasetDict(ds_new)
ds_new.push_to_hub(OUTPUT_NAME)

<>:14: SyntaxWarning: invalid escape sequence '\o'
<>:17: SyntaxWarning: invalid escape sequence '\o'
<>:14: SyntaxWarning: invalid escape sequence '\o'
<>:17: SyntaxWarning: invalid escape sequence '\o'
C:\Users\valte\AppData\Local\Temp\ipykernel_28336\2551950308.py:14: SyntaxWarning: invalid escape sequence '\o'
  with open("optimization_results\optimization_results_train.json") as f:
C:\Users\valte\AppData\Local\Temp\ipykernel_28336\2551950308.py:17: SyntaxWarning: invalid escape sequence '\o'
  with open("optimization_results\optimization_results_test.json") as f:
c:\Users\valte\OneDrive - University of Helsinki\Desktop\rl-quantum\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\valte\OneDrive - University of Helsinki\Desktop\rl-quantum\myenv\Lib\site-packages\qiskit_ibm_provider\api\session.py:1

CommitInfo(commit_url='https://huggingface.co/datasets/valterUo/graph-data-quantum-optimized/commit/386c2bb0c9f074fc47477b2b5cb3405e305add2f', commit_message='Upload dataset', commit_description='', oid='386c2bb0c9f074fc47477b2b5cb3405e305add2f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/valterUo/graph-data-quantum-optimized', endpoint='https://huggingface.co', repo_type='dataset', repo_id='valterUo/graph-data-quantum-optimized'), pr_revision=None, pr_num=None)